# 第 3 周 —— 用开源工具做合成数据生成器

## 概述（练习目标）

本笔记本演示如何组合三类工具，搭一个接近「可落地」的合成数据流水线：

- **Faker**：生成真实感的结构化字段（姓名、邮箱等）
- **Ollama**：用**本地** LLM 生成有上下文的文本（例如职位相关 Bio）
- **Pandas**：整理成表、做校验、导出 CSV

## 用例

为「员工数据集」生成与**职位 / 资历**匹配的 AI 简介，适合做测试数据、演示环境填充，或隐私友好的合成替代数据。

## 怎么跑

1. 本机已安装并启动 **Ollama**，且已拉取 `CONFIG["LLM_MODEL"]`（默认 `llama3`）
2. 从上到下运行：安装 → 导入 → 配置 → 定义引擎 → 生成 → 校验 → 导出
3. 想换规模或模型：改 `CONFIG` 后重新跑生成相关单元格


## 第 1 步：设置和导入

先安装所需第三方库，再导入到当前笔记本内核，供后续单元格使用。


In [ ]:
# ========== 安装依赖：Faker / Ollama 客户端 / Pandas / tqdm ==========
# notebook shell magic：在系统环境安装包；-q 表示安静模式（少打印）
!pip install faker ollama pandas tqdm -q 


In [ ]:
# ========== 导入库：表格、假数据、本地 LLM、进度条、随机与类型标注 ==========

# 导入 pandas：用 DataFrame 承载最终合成表
import pandas as pd
# 从 faker 导入 Faker：生成姓名、邮箱等假但逼真的字段
from faker import Faker
# 导入 ollama：Python 客户端，调用本机 Ollama 的 generate 接口
import ollama
# 从 tqdm 导入进度条：长时间批量生成时给用户反馈
from tqdm import tqdm
# 导入标准库 random：资历抽样、职位抽样、经验年数区间
import random
# 从 typing 导入类型标注：Dict / List / Any，方便读类方法签名
from typing import Dict, List, Any
# 导入 warnings：后面统一忽略告警，保持输出干净（按原文）
import warnings

# 忽略各类 Warning，避免刷屏（教学环境常见写法）
warnings.filterwarnings('ignore')

# 运行时提示保持英文原样
print("✓ All libraries imported successfully!")


## 第 2 步：配置

把数据集规模、字段来源（faker / custom / llm）、模型名、导出路径等集中写进一个 `CONFIG` 字典，后面只改这一处即可。


In [ ]:
# ========== 集中配置：行数、schema、LLM、职位池、资历、导出文件名 ==========

# 配置字典：生成参数 + 字段来源声明 + 业务选项
CONFIG = {
    # 发电参数（生成规模与批大小）
    "NUM_ROWS": 50,
    # 分批生成，降低单次循环压力（注释原文 Process in batches... 保留含义）
    "BATCH_SIZE": 10,  # Process in batches to manage memory
    
    # 模式定义：每个字段由谁负责生成
    "SCHEMA": {
        "Name": "faker",           # Generated by Faker
        "Email": "faker",          # Generated by Faker
        "Job Title": "faker",      # Generated by Faker
        "Seniority": "custom",     # Custom logic (Junior, Mid, Senior)
        "Years of Experience": "custom",  # Custom logic (based on seniority)
        "AI Generated Bio": "llm"  # Generated by Ollama LLM
    },
    
    # 法学硕士配置（本地 Ollama 模型名与采样温度）
    "LLM_MODEL": "llama3",  # Options: llama3, mistral, phi, etc.
    "LLM_TEMPERATURE": 0.7,
    
    # 职位名称选项：Job Title 从该列表随机抽
    "JOB_TITLES": [
        "Software Engineer", "Data Scientist", "Product Manager",
        "UX Designer", "DevOps Engineer", "Marketing Manager",
        "Sales Representative", "HR Manager", "Financial Analyst"
    ],
    
    # 资历级别：业务逻辑会据此约束「工作年限」区间
    "SENIORITY_LEVELS": ["Junior", "Mid", "Senior"],
    
    # 导出配置：最终 CSV 文件名（相对当前工作目录）
    "OUTPUT_FILE": "synthetic_data.csv"
}

# 打印关键配置摘要，确认加载成功
print("Configuration loaded:")
print(f"  • Generating {CONFIG['NUM_ROWS']} rows")
print(f"  • Using LLM model: {CONFIG['LLM_MODEL']}")
print(f"  • Schema fields: {', '.join(CONFIG['SCHEMA'].keys())}")


## 步骤 3：核心生成器类

`SyntheticDataEngine` 协调整条流水线：

- 用 **Faker** 填标准字段（姓名、邮箱等）
- 连 **Ollama** 生成 AI 文本字段
- 用**业务规则**保证资历与年限一致
- **分批**生成，并用进度条反馈


In [ ]:
# ========== 核心类 SyntheticDataEngine：Faker + 业务规则 + Ollama Bio ==========

class SyntheticDataEngine:
    """
    A comprehensive synthetic data generator that combines traditional fake data
    with AI-generated content using local LLMs via Ollama.
    """
    
    def __init__(self, config: Dict[str, Any]):
        """
        Initialize the data generator with configuration.
        
        Args:
            config: Configuration dictionary containing schema, LLM settings, etc.
        """
        # 保存配置引用，后续方法都从 self.config 读参数
        self.config = config
        # 创建 Faker 实例：负责 name/email 等
        self.faker = Faker()
        # 固定 Faker 种子，便于复现同一批「假」数据
        Faker.seed(42)  # For reproducibility
        # 同步固定 random 种子（资历/职位/年限抽样可复现）
        random.seed(42)
        
        # 初始化成功提示（字符串保持原样）
        print(f"🚀 SyntheticDataEngine initialized")
        print(f"   Model: {config['LLM_MODEL']}")
        
    def _generate_faker_field(self, field_name: str) -> str:
        """Generate data using Faker library."""
        # 按字段名分派到不同 Faker / 随机逻辑
        if field_name == "Name":
            return self.faker.name()
        elif field_name == "Email":
            return self.faker.email()
        elif field_name == "Job Title":
            # 职位不走 Faker 职业库，而从 CONFIG 的 JOB_TITLES 抽
            return random.choice(self.config["JOB_TITLES"])
        # 未知字段名：返回空串，避免 KeyError
        return ""
    
    def _apply_business_logic(self, row_data: Dict[str, Any]) -> Dict[str, Any]:
        """
        Apply business rules to ensure data consistency.
        
        Business Rules:
        1. Junior: 0-3 years of experience
        2. Mid: 3-7 years of experience
        3. Senior: 7+ years of experience
        """
        # 随机抽一个资历级别，写入当前行
        seniority = random.choice(self.config["SENIORITY_LEVELS"])
        row_data["Seniority"] = seniority
        
        # 资历 → 工作年限闭区间（业务一致性的关键约束）
        if seniority == "Junior":
            years_exp = random.randint(0, 3)
        elif seniority == "Mid":
            years_exp = random.randint(3, 7)
        else:  # Senior
            years_exp = random.randint(7, 15)
        
        row_data["Years of Experience"] = years_exp
        
        return row_data
    
    def _generate_ai_bio(self, job_title: str, seniority: str, years_exp: int) -> str:
        """
        Generate a contextual biography using Ollama LLM.
        
        Args:
            job_title: The person's job title
            seniority: Seniority level (Junior/Mid/Senior)
            years_exp: Years of experience
            
        Returns:
            AI-generated professional biography
        """
        # 英文 prompt 必须保留：描述资历/职位/年限，要求 2–3 句且不写姓名
        prompt = f"""Write a brief professional bio (2-3 sentences) for a {seniority} {job_title} with {years_exp} years of experience. 
Make it realistic and professional. Focus on skills and achievements relevant to their role.
Do not include a name."""
        
        try:
            # 调用本地 Ollama generate；model / temperature / num_predict 来自配置
            response = ollama.generate(
                model=self.config["LLM_MODEL"],
                prompt=prompt,
                options={
                    "temperature": self.config["LLM_TEMPERATURE"],
                    "num_predict": 100  # Limit response length
                }
            )
            # Ollama 返回字典，文本在 'response' 键
            return response['response'].strip()
        except Exception as e:
            # LLM 失败时的英文回退句式（保持原样，保证流水线不中断）
            return f"Experienced {seniority} {job_title} with {years_exp} years in the industry."
    
    def generate_row(self) -> Dict[str, Any]:
        """
        Generate a single row of synthetic data.
        
        Returns:
            Dictionary containing all fields for one record
        """
        # 先准备空行字典，再按 schema 分三步填充
        row_data = {}
        
        # 第1步：SCHEMA 里标记为 faker 的字段
        for field_name, field_type in self.config["SCHEMA"].items():
            if field_type == "faker":
                row_data[field_name] = self._generate_faker_field(field_name)
        
        # 第 2 步：自定义业务字段（Seniority / Years of Experience）
        row_data = self._apply_business_logic(row_data)
        
        # 第 3 步：若 schema 含 AI Generated Bio，则调本地 LLM
        if "AI Generated Bio" in self.config["SCHEMA"]:
            row_data["AI Generated Bio"] = self._generate_ai_bio(
                row_data["Job Title"],
                row_data["Seniority"],
                row_data["Years of Experience"]
            )
        
        return row_data
    
    def generate_batch(self, batch_size: int) -> List[Dict[str, Any]]:
        """
        Generate a batch of synthetic data rows.
        
        Args:
            batch_size: Number of rows to generate
            
        Returns:
            List of dictionaries, each representing a row
        """
        # 列表推导：连续调用 generate_row batch_size 次
        return [self.generate_row() for _ in range(batch_size)]
    
    def generate_dataset(self) -> pd.DataFrame:
        """
        Generate the complete dataset with progress tracking.
        
        Returns:
            Pandas DataFrame containing all synthetic data
        """
        # 总行数与批大小来自 CONFIG
        num_rows = self.config["NUM_ROWS"]
        batch_size = self.config["BATCH_SIZE"]
        
        # 累积所有行字典
        all_data = []
        # 向上取整批次数：(n + b - 1) // b
        num_batches = (num_rows + batch_size - 1) // batch_size
        
        print(f"\n📊 Generating {num_rows} rows in {num_batches} batches...")
        
        # tqdm 进度条：按「行」更新
        with tqdm(total=num_rows, desc="Generating data", unit="rows") as pbar:
            for batch_num in range(num_batches):
                # 本批实际行数：不超过剩余未生成行数
                remaining_rows = num_rows - len(all_data)
                current_batch_size = min(batch_size, remaining_rows)
                
                # 生成一批并追加
                batch_data = self.generate_batch(current_batch_size)
                all_data.extend(batch_data)
                
                # 推进进度条
                pbar.update(current_batch_size)
        
        # list[dict] → DataFrame
        df = pd.DataFrame(all_data)
        print(f"✓ Dataset generation complete! Shape: {df.shape}")
        
        return df

print("✓ SyntheticDataEngine class defined")


## 步骤 4：生成数据集

实例化 `SyntheticDataEngine`，调用 `generate_dataset()`；过程中会看到 tqdm 进度条。


In [ ]:
# ========== 跑起来：用 CONFIG 初始化引擎，再生成完整 DataFrame ==========

# 把上面的 CONFIG 注入引擎
engine = SyntheticDataEngine(CONFIG)

# 按 NUM_ROWS / BATCH_SIZE 批量生成，返回 df
df = engine.generate_dataset()


## 步骤 5：数据验证和探索

检查生成结果的形状、类型、统计量，并核对「资历 ↔ 年限」业务规则是否生效。


In [ ]:
# ========== 5.1 预览：看前 5 行长什么样 ==========
### 5.1 预览前 5 行

# 打印说明后，用 DataFrame.head() 展示前几行（Jupyter 会显示表格）
print("📋 First 5 rows of generated data:\n")
df.head()


In [ ]:
# ========== 5.2 数据集信息：行数、列数、dtype、内存占用 ==========
### 5.2 数据集信息

print("ℹ️  Dataset Information:\n")
# 行数 = 记录条数
print(f"Total Rows: {len(df)}")
# 列数 = 字段个数
print(f"Total Columns: {len(df.columns)}")
print(f"\nColumn Names and Types:")
# dtypes：每列的 pandas 数据类型
print(df.dtypes)
# deep=True 统计对象列真实内存，再换成 KB
print(f"\nMemory Usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")


In [ ]:
# ========== 5.3 统计摘要：describe(include='all') 看分布概况 ==========
### 5.3 统计总结

print("📊 Statistical Summary:\n")
# include='all'：数值列与非数值列都给出摘要统计
df.describe(include='all')


### 5.4 业务逻辑验证

核对业务规则是否正确落在数据上：

- **Junior**：工作经验应在 0–3 年
- **Mid**：工作经验应在 3–7 年
- **Senior**：工作经验应 ≥ 7 年


In [ ]:
# ========== 按资历聚合年限，并做三条规则的布尔校验 ==========

print("🔍 Business Logic Validation:\n")

# groupby Seniority 后对 Years of Experience 算 min/max/mean/count
validation = df.groupby('Seniority')['Years of Experience'].agg(['min', 'max', 'mean', 'count'])
print(validation)

# 逐条规则检查：Junior 最大年限 ≤ 3；Mid 落在 [3,7]；Senior 最小 ≥ 7
print("\n✓ Validation Results:")
junior_valid = df[df['Seniority'] == 'Junior']['Years of Experience'].max() <= 3
mid_valid = (df[df['Seniority'] == 'Mid']['Years of Experience'].min() >= 3) and \
            (df[df['Seniority'] == 'Mid']['Years of Experience'].max() <= 7)
senior_valid = df[df['Seniority'] == 'Senior']['Years of Experience'].min() >= 7

# 用三元表达式打印 PASS / FAIL（英文标记保持原样）
print(f"  Junior rules: {'✓ PASS' if junior_valid else '✗ FAIL'}")
print(f"  Mid rules: {'✓ PASS' if mid_valid else '✗ FAIL'}")
print(f"  Senior rules: {'✓ PASS' if senior_valid else '✗ FAIL'}")


### 5.5 AI 生成的 Bio 示例

从 Junior / Mid / Senior 各抽一条，人工扫一眼 Ollama 生成的简介质量与相关性。


In [ ]:
# ========== 抽样展示：三个资历层级各取 1 行 Bio ==========

print("🤖 Sample AI-Generated Biographies:\n")

# 按资历循环；sample(1) 随机抽一行，iloc[0] 取出 Series
for seniority in ['Junior', 'Mid', 'Senior']:
    sample = df[df['Seniority'] == seniority].sample(1).iloc[0]
    print(f"{'='*70}")
    print(f"Name: {sample['Name']}")
    print(f"Title: {sample['Seniority']} {sample['Job Title']}")
    print(f"Experience: {sample['Years of Experience']} years")
    print(f"\nBio: {sample['AI Generated Bio']}")
    print()


## 步骤 6：导出数据

把生成好的 DataFrame 写成 CSV，供其它工具、测试脚本或下游 notebook 使用。


In [ ]:
# ========== 导出 CSV，并确认文件已落盘 ==========

# 从 CONFIG 读取输出文件名（与生成配置保持单一来源）
output_file = CONFIG["OUTPUT_FILE"]
# index=False：不把 DataFrame 行号写入 CSV
df.to_csv(output_file, index=False)

print(f"✓ Dataset exported successfully!")
print(f"  File: {output_file}")
print(f"  Size: {len(df)} rows × {len(df.columns)} columns")

# 验证文件已创建：存在则打印字节大小（KB）
import os
if os.path.exists(output_file):
    file_size = os.path.getsize(output_file) / 1024
    print(f"  File size: {file_size:.2f} KB")


## 总结和要点

### 我们建造了什么

这个笔记本演示了一条可复用的合成数据流水线，它组合了：

1. **传统假数据**（Faker）填结构化字段
2. **本地 AI 文本**（Ollama）生成有上下文的 Bio
3. **业务逻辑**保证资历与年限一致
4. **批处理**控制单次工作量
5. **进度条**给长时间任务反馈

### 架构亮点

#### 1. 模块化设计

- 配置（`CONFIG`）与逻辑（`SyntheticDataEngine`）分离
- 类可复用，便于加字段或加规则

#### 2. 数据质量

- 业务规则约束逻辑一致性
- 校验单元格确认规则是否生效
- LLM 文本提升「看起来像真人」的真实感

#### 3. 可扩展性

- 批处理减轻长时间循环的心理负担与资源尖峰
- 提高 `NUM_ROWS` 即可放大
- tqdm 让你知道还要等多久

#### 4. 本地优先

- Ollama 本地推理：更利于隐私与成本控制
- 不依赖云端 API Key（Faker / Pandas 亦本地）
- 模型与温度都在配置里可调

### 用例

- **测试**：给应用造逼真测试数据
- **训练**：为 ML 流程准备合成样本
- **演示**：填充 demo 环境
- **隐私**：用合成数据代替敏感生产数据

### 后续步骤

- 增加字段（电话、地址、部门）
- 写更复杂的跨字段规则
- 加数据质量检查（邮箱格式等）
- 生成关联表（员工 → 项目 → 任务）
- 可视化分布
- 导出 JSON / Parquet / SQL 等格式


## 奖励：快速重跑

想换一套数据集？改 `CONFIG`（行数、模型、输出文件名），再重新运行「生成」相关单元格即可。


In [ ]:
# ========== 可选实验：改 CONFIG 后重新生成（默认整段注释掉） ==========
# 示例：使用不同的设置生成更大的数据集
# 取消注释并运行尝试：

# 配置[“NUM_ROWS”] = 100
# CONFIG["LLM_MODEL"] = "米斯特拉尔"
# CONFIG["OUTPUT_FILE"] = "synthetic_data_large.csv"
# 
# engine_v2 = SyntheticDataEngine(CONFIG)
# df_v2 = engine_v2.generate_dataset()
# df_v2.to_csv(CONFIG["OUTPUT_FILE"]，索引=False)
# print(f"✓ 使用 {len(df_v2)} 行创建的新数据集！")

# 提示：上面多行仍是注释；取消注释前请确认标识符与标点是合法 Python
print("💡 Tip: Uncomment the code above to generate a different dataset")
